# Ingestão Incremental de Dados

## 📊 Conceito Fundamental

**Ingestão Incremental** = Carregar apenas arquivos **novos** desde a última ingestão.

**Benefícios:**
- Reduz processamento redundante
- Evita reprocessar dados já carregados
- Otimiza custos de computação

**Dois mecanismos principais no Databricks:**

```
┌─────────────────────────────────────────────────────────────┐
│                  Ingestão Incremental                       │
├─────────────────────────┬───────────────────────────────────┤
│       COPY INTO         │          Auto Loader              │
│    (Comando SQL)        │    (Structured Streaming)         │
│   Milhares de arquivos  │     Milhões de arquivos           │
└─────────────────────────┴───────────────────────────────────┘
```

---

## 📋 COPY INTO

### O que é?
- **Comando SQL** para carregamento incremental e idempotente
- Arquivos já carregados são **automaticamente ignorados**
- Mesmo se os arquivos foram modificados após o carregamento

### Sintaxe Básica

```sql
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = <formato>
FORMAT_OPTIONS (<opções de formato>)
COPY_OPTIONS (<opções de cópia>);
```

### Exemplo Prático

```sql
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'delimiter' = '|',
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);
```

### Formatos Suportados

| Formato | Extensão |
|---------|----------|
| CSV | .csv |
| JSON | .json |
| PARQUET | .parquet |
| AVRO | .avro |
| ORC | .orc |
| TEXT | .txt |
| BINARYFILE | binários |

### Opções Importantes

| Categoria | Opção | Descrição |
|-----------|-------|-----------|
| FORMAT_OPTIONS | `delimiter` | Delimitador para CSV |
| FORMAT_OPTIONS | `header` | Se arquivo tem cabeçalho |
| FORMAT_OPTIONS | `inferSchema` | Inferir tipos automaticamente |
| COPY_OPTIONS | `mergeSchema` | Mesclar esquemas automaticamente |

> 💡 **Para prova:** COPY INTO é **idempotente** - executar múltiplas vezes não duplica dados!

---

## 🚀 Auto Loader

### O que é?
- Baseado em **Structured Streaming**
- Usa formato `cloudFiles`
- Processa **bilhões de arquivos** eficientemente
- Suporta ingestão **near real-time** de milhões de arquivos/hora

### Checkpointing no Auto Loader

| Função | Descrição |
|--------|-----------|
| **Armazenamento de metadados** | Guarda info dos arquivos descobertos |
| **Garantia exactly-once** | Cada arquivo processado uma única vez |
| **Tolerância a falhas** | Retoma do ponto onde parou |

> ⚠️ **Crítico para prova:** Metadados são armazenados em **RocksDB** no checkpoint location!

### Sintaxe Básica (PySpark)

```python
spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", <formato>) \
    .load('/path/to/files') \
    .writeStream \
    .option("checkpointLocation", <checkpoint_dir>) \
    .table(<table_name>)
```

### Com Inferência de Schema

```python
spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", <formato>) \
    .option("cloudFiles.schemaLocation", <schema_dir>) \
    .load('/path/to/files') \
    .writeStream \
    .option("checkpointLocation", <checkpoint_dir>) \
    .option("mergeSchema", "true") \
    .table(<table_name>)
```

### Modos de Detecção de Arquivos

| Modo | Descrição | Quando usar |
|------|-----------|-------------|
| **Directory Listing** | Escaneia diretórios periodicamente | Workloads menores |
| **File Notification** | Usa eventos da cloud (Event Grid/SQS) | Workloads grandes, milhões de arquivos |

### Formatos Suportados pelo Auto Loader

JSON, CSV, XML, PARQUET, AVRO, ORC, TEXT, BINARYFILE

### Vantagens do Auto Loader

| Vantagem | Descrição |
|----------|-----------|
| **Escalabilidade** | Descobre bilhões de arquivos eficientemente |
| **Performance** | Custo escala com nº de arquivos, não diretórios |
| **Schema Evolution** | Detecta drift, notifica mudanças, resgata dados |
| **Custo** | File notification evita listagem de diretórios |

---

## ⚖️ COPY INTO vs Auto Loader

| Critério | COPY INTO | Auto Loader |
|----------|-----------|-------------|
| **Tipo** | Comando SQL | Structured Streaming |
| **Volume** | **Milhares** de arquivos | **Milhões** de arquivos |
| **Escala** | Menos eficiente | Altamente eficiente |
| **Schema Evolution** | Limitado | Completo |
| **Real-time** | Batch | Near real-time |
| **Complexidade** | Simples | Mais configuração |

### Quando usar cada um?

```
┌─────────────────────────────────────────────────────────────┐
│  Poucos arquivos (< 10.000)?  ───────►  COPY INTO           │
│  Muitos arquivos (milhões)?   ───────►  Auto Loader         │
│  Precisa de near real-time?   ───────►  Auto Loader         │
│  Schema muda frequentemente?  ───────►  Auto Loader         │
│  Simplicidade é prioridade?   ───────►  COPY INTO           │
└─────────────────────────────────────────────────────────────┘
```

---



## ✅ Checklist para Prova

- [ ] **COPY INTO** é comando SQL, **Auto Loader** usa Structured Streaming
- [ ] COPY INTO para **milhares** de arquivos, Auto Loader para **milhões**
- [ ] Ambos são **idempotentes** (não duplicam dados)
- [ ] Auto Loader usa formato **`cloudFiles`**
- [ ] Checkpoint armazena metadados em **RocksDB**
- [ ] Auto Loader oferece **exactly-once** guarantee
- [ ] **Schema evolution** é melhor no Auto Loader
- [ ] File notification mode usa **Event Grid** (Azure) ou **SQS** (AWS)
- [ ] `cloudFiles.schemaLocation` para inferência de schema
- [ ] `mergeSchema = true` para evolução automática

---

## 🔗 Referências (Documentação em Português)

- [O que é o Carregador Automático (Auto Loader)?](https://learn.microsoft.com/pt-br/azure/databricks/ingestion/cloud-object-storage/auto-loader/)
- [Tutorial: COPY INTO com Spark SQL](https://learn.microsoft.com/pt-pt/azure/databricks/ingestion/cloud-object-storage/copy-into/tutorial-notebook)
- [COPY INTO - Referência SQL](https://docs.databricks.com/pt/sql/language-manual/delta-copy-into.html)
- [Configurar Auto Loader para produção](https://learn.microsoft.com/pt-br/azure/databricks/ingestion/cloud-object-storage/auto-loader/production)